# Finding one vibration mode by counting eigenvalues
Change the mass of one bead in a fixed-end chain. Use a three-term inertia recurrence to enclose the chosen eigenvalue without computing an eigenvector. Frequencies are measured in units of the frequency of a single spring and unit mass. The calculation runs with NumPy and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

beads = 15
middle_mass = 3.0
mode = 5                   # one-based, starting at the lowest frequency
assert 3 <= beads <= 100 and 0.2 <= middle_mass <= 10 and 1 <= mode <= beads
masses = np.ones(beads)
masses[beads // 2] = middle_mass
diag = 2 / masses
off = -1 / np.sqrt(masses[:-1] * masses[1:])
A = np.diag(diag) + np.diag(off, 1) + np.diag(off, -1)

## Count signed pivots
In exact arithmetic each negative pivot contributes one eigenvalue below the shift as long as no pivot vanishes. For the unit-scale matrices in this notebook a vanishing pivot is replaced by a tiny negative pivot. This assigns an exact tie to the lower spectrum. The common test below also checks the count at an eigenvalue analytically known to be exact.

In [ ]:
def inertia_count(a, b, shift):
    floor = np.finfo(float).tiny * max(1.0, np.max(b*b, initial=0.0))
    pivot = a[0] - shift
    negatives = 0
    for i in range(len(a)):
        if abs(pivot) < floor:
            pivot = -floor
        negatives += int(pivot < 0)
        if i + 1 < len(a):
            pivot = a[i+1] - shift - (b[i]*b[i])/pivot
    return negatives

assert inertia_count(np.array([2., 2., 2.]), np.array([-1., -1.]), 2.0) == 2

def enclosing_interval(a, b, k, rounds=44):
    radii = np.r_[0., abs(b)] + np.r_[abs(b), 0.]
    lower = float(min(a-radii) - 1)
    upper = float(max(a+radii) + 1)
    history = []
    for _ in range(rounds):
        center = (lower+upper)/2
        history.append([lower, upper])
        if inertia_count(a, b, center) < k:
            lower = center
        else:
            upper = center
    return lower, upper, np.array(history)

left, right, history = enclosing_interval(diag, off, mode)
reference = np.linalg.eigvalsh(A)
assert left - 5e-13 <= reference[mode-1] <= right + 5e-13
print(f"Mode {mode}: eigenvalue in [{left:.12f}, {right:.12f}]")
print(f"Angular frequency = {np.sqrt((left+right)/2):.8f} in spring–mass units.")

## Locate the step; watch its bracket contract
The staircase changes height when the trial shift crosses an eigenvalue. The second panel shows the bracket width after each iteration. Try a lighter bead. Does every mode move by the same amount?

In [ ]:
fig, (spectrum, convergence) = plt.subplots(1, 2, figsize=(10, 4.0), layout="constrained")
shifts = np.linspace(-0.15, reference[-1]+0.2, 500)
spectrum.step(shifts, [inertia_count(diag, off, x) for x in shifts], where="post", color="#145c71")
spectrum.axvline(reference[mode-1], color="#b94626", ls="--", label=f"mode {mode}")
spectrum.set(xlabel="Shift in squared-frequency units", ylabel="Eigenvalue count", title="Count the negative pivots")
spectrum.legend()
convergence.semilogy(range(1, len(history)+1), history[:,1]-history[:,0], color="#145c71")
convergence.set(xlabel="Bisection iteration", ylabel="Bracket width", title="One reliable binary digit per split")
for ax in (spectrum, convergence): ax.grid(alpha=.2)
plt.show()

## Independent checks
With unit masses the eigenvalues follow from sine modes. An installed OneSimulation environment can cross-check the modified chain through its dense symmetric eigensolver.

In [ ]:
k = np.arange(1, beads+1)
uniform = np.diag(np.full(beads,2.0)) + np.diag(-np.ones(beads-1),1) + np.diag(-np.ones(beads-1),-1)
assert np.max(np.abs(np.linalg.eigvalsh(uniform) - 4*np.sin(k*np.pi/(2*(beads+1)))**2)) < 1e-12
try:
    from onesim.numerics.linear.matrices.dense_linear_algebra import symmetric_eigenvalues
except ImportError:
    print("Sine-spectrum and NumPy reference checks passed. OneSimulation comparison is available locally.")
else:
    independent = symmetric_eigenvalues(A)
    print("OneSimulation squared frequency:", independent[mode-1])
    assert np.allclose(independent, reference)